# Circuit Motif Discovery — Full Colab Pipeline

Run all steps from environment setup to final figures.

In [ ]:
# Cell 1: Setup (Drive/upload-first, no GitHub required)
!nvidia-smi

from pathlib import Path
import os

# Option A (recommended): project folder already in Drive
# Expected location: /content/drive/MyDrive/circuit-motif-discovery
from google.colab import drive
drive.mount('/content/drive')

repo_dir = Path('/content/drive/MyDrive/circuit-motif-discovery')
zip_path = Path('/content/drive/MyDrive/circuit-motif-discovery.zip')

# Option B: if only zip exists in Drive, unzip it
if (not repo_dir.exists()) and zip_path.exists():
    !unzip -q /content/drive/MyDrive/circuit-motif-discovery.zip -d /content/drive/MyDrive/

# Option C: if you uploaded folder/zip to /content directly
if not repo_dir.exists():
    local_repo = Path('/content/circuit-motif-discovery')
    local_zip = Path('/content/circuit-motif-discovery.zip')
    if (not local_repo.exists()) and local_zip.exists():
        !unzip -q /content/circuit-motif-discovery.zip -d /content/
    repo_dir = local_repo

assert repo_dir.exists(), (
    'Project folder not found. Upload circuit-motif-discovery folder (or zip) '
    'to Drive or /content first.'
)

%cd {repo_dir}
!bash setup_colab.sh

import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# Cell 2: Quick Test (optional, ~10 minutes)
!python scripts/run_full_pipeline.py --quick

In [ ]:
# Cell 3: Generate Graphs
!python scripts/01_generate_graphs.py \
  --prompt_file prompts/prompt_corpus.json \
  --output_dir data/graphs \
  --max_prompts_per_family 50

import json
from pathlib import Path
summary = json.loads(Path('data/graphs/summary.json').read_text())
print(summary)

In [ ]:
# Cell 4: Convert to PyG
!python scripts/02_convert_to_pyg.py \
  --input_dir data/graphs \
  --output_path data/circuit_dataset.pt

import torch
dataset = torch.load('data/circuit_dataset.pt', weights_only=False)
print('Num graphs:', len(dataset))
print('Sample graph:', dataset[0])

In [ ]:
# Cell 5: Train GCL
!python scripts/03_train_contrastive.py \
  --dataset_path data/circuit_dataset.pt \
  --output_dir checkpoints/

import torch
ckpt = torch.load('checkpoints/best.pt', map_location='cpu', weights_only=False)
loss_history = ckpt.get('history', [])
print('Best epoch:', ckpt.get('epoch'))
print('Best loss:', ckpt.get('loss'))

import matplotlib.pyplot as plt
plt.figure(figsize=(6, 4))
plt.plot(loss_history)
plt.title('Contrastive Training Loss')
plt.xlabel('Epoch')
plt.ylabel('NT-Xent Loss')
plt.show()

In [ ]:
# Cell 6: Evaluate
!python scripts/04_evaluate.py \
  --dataset_path data/circuit_dataset.pt \
  --checkpoint checkpoints/best.pt \
  --output_dir results/

from IPython.display import Image, display
for p in [
    'results/demo1_umap.png',
    'results/demo2_retrieval.png',
    'results/demo3_cluster_motifs.png',
    'results/demo4_metrics.png',
]:
    display(Image(p))

import json
print(json.dumps(json.load(open('results/metrics_summary.json')), indent=2))

In [ ]:
# Cell 7: Save Results
from pathlib import Path

# Save results to Drive for persistence
drive_out = Path('/content/drive/MyDrive/circuit-motif-discovery-results')
drive_out.mkdir(parents=True, exist_ok=True)
!cp -r results "{drive_out}"

print('Artifacts in results/:')
for p in sorted(Path('results').glob('*')):
    print('-', p)
print('\nCopied to:', drive_out)